In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2

In [2]:
# Basic experiment settings
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)


cudnn.deterministic = False
cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Declare directories
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

# Experiment parameters
model_b_img_size = 384
model_a_img_size = 384
batch_size = 16
num_workers = 4
num_classes = 5
tta = False

# Currently testing vision transformers so check its architecture
model_a = torch.load("/kaggle/input/vit-v1/vit_v1.pt", map_location=device)
model_b = torch.load(
    "/kaggle/input/vit-boosted/vit_boosted.pt", map_location=device
)
linear_head = torch.load("/kaggle/input/linear-head/linear_cls.pt", map_location=device)

cpu


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/vit-v1/vit_v1.pt'

In [3]:
# Test dataset without labels
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Args:
        data_dir: base directory to the images.
        img_label_df: dataframe containing image-label pairs.
        transforms: set of transforms to be used.
    """

    def __init__(
        self,
        data_dir,
        model_a_size,
        model_b_size,
        transform=None,
        ttas=None,
        img_size=384,
    ):
        super().__init__(root=data_dir)

        self.transform = transform
        self.images = os.listdir(data_dir)
        self.ttas = ttas
        self.cc = v2.CenterCrop((600, 600))
        self.resize_model_a = v2.Resize(
            (model_a_size, model_a_size), interpolation=InterpolationMode.BICUBIC
        )

        self.resize_model_b = v2.Resize(
            (model_b_size, model_b_size), interpolation=InterpolationMode.BICUBIC
        )

    def __getitem__(self, idx):
        filename = self.images[idx]
        img = Image.open(os.path.join(self.root, filename))
        img = self.cc(img)
        model_a_img = self.resize_model_a(img)
        model_b_img = self.resize_model_b(img)

        if self.ttas is not None and self.transform is not None:
            model_a_img = [self.transform(t(model_a_img)) for t in self.ttas]
            model_b_img = [self.transform(t(model_b_img)) for t in self.ttas]

        elif self.transform:
            model_a_img = self.transform(model_a_img)
            model_b_img = self.transform(model_b_img)

        return model_a_img, model_b_img, filename

    def __len__(self):
        return len(self.images)

In [4]:
# Create the set of transforms for test time
test_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

if tta:
    ttas = [
        #         v2.RandomResizedCrop((img_size, img_size), (0.5, 1)),
        v2.RandomRotation(180),
        v2.RandomVerticalFlip(1),
#         v2.RandomAffine(180),
        v2.RandomPerspective(p=1),
    ]
else:
    ttas = None

# Dataset and dataloader
test_dataset = CassavaDataset(
    test_dir, model_a_img_size, model_b_img_size, transform=test_transforms, ttas=ttas
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

# Adjust unnormalized outputs
normalizer = torch.nn.Softmax(dim=1)

In [5]:
# Perform inference
all_names = []
all_preds = []

model_a.eval()
model_b.eval()
linear_head.eval()

with torch.no_grad():
    test_loss = 0

    for batch_idx, (model_a_inputs, model_b_inputs, filenames) in enumerate(
        test_loader
    ):
        batch_size = len(filenames)

        if tta:
            model_a_inputs, model_b_inputs, filenames = (
                torch.cat(model_a_inputs, dim=0).to(device),
                torch.cat(model_b_inputs, dim=0).to(device),
                list(filenames),
            )

            model_a_outputs = model_a(model_a_inputs)
            model_b_outputs = model_b(model_b_inputs)

            # Find the average prediction logits by each model
            model_a_batch_logits = torch.stack(
                torch.split(model_a_outputs, batch_size), dim=0
            )
            model_a_mean_logits = torch.mean(model_a_batch_logits, dim=0)

            model_b_batch_logits = torch.stack(
                torch.split(model_b_outputs, batch_size), dim=0
            )
            model_b_mean_logits = torch.mean(model_b_batch_logits, dim=0)

            #             logit_inputs = torch.cat([model_a_mean_logits, model_b_mean_logits], dim=1)
            #             outputs = linear_head(logit_inputs)

            outputs = (0.9 * model_a_mean_logits + 0.1 * model_b_mean_logits) / 2

            mean_preds = normalizer(outputs)
            pred_labels = torch.argmax(mean_preds, 1).tolist()

        else:
            # The same thing but without the troublesome stuff
            model_a_inputs, model_b_inputs, filenames = (
                model_a_inputs.to(device),
                model_b_inputs.to(device),
                list(filenames),
            )

            model_a_outputs = model_a(model_a_inputs)
            model_b_outputs = model_b(model_b_inputs)

            #             logit_inputs = torch.cat([model_a_outputs, model_b_outputs], dim=1)
            #             outputs = linear_head(logit_inputs)
            outputs = (0.92 * model_a_outputs + 0.08 * model_b_outputs) / 2

            preds = normalizer(outputs)
            pred_labels = torch.argmax(preds, 1).tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)

NameError: name 'model_a' is not defined

In [6]:
# Create submission
my_submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
my_submission.to_csv("submission.csv", index=False)
my_submission

,image_id,label
